In [ ]:
import coffea
import hist
import pickle
from coffea import util
import matplotlib.pyplot as plt
import itertools
import os, sys
import mplhep as hep
import numpy as np
hep.style.use("CMS")

sys.path.append('../python/')
from functions import getCoffeaFilenames, getHist

useOldHTcut = False
oldHTstr = ''
if useOldHTcut:
    oldHTstr = '_oldHTcut'

In [ ]:
f1 = util.load('../outputs/TTbar_2016APV_700to1000'+oldHTstr+'.coffea')
f2 = util.load('../outputs/TTbar_2016APV_1000toInf'+oldHTstr+'.coffea')
# f3 = util.load('../outputs/JetHT_2016APVB.coffea')

In [ ]:
ttagcats = ['at', 'pret', '2t']
btagcats = ["0b", "1b", "2b"]
ycats = ['cen', 'fwd']                                                                                                          
anacats = [ t+b+y for t,b,y in itertools.product( ttagcats, btagcats, ycats) ]
label_dict = {i: label for i, label in enumerate(anacats)}

In [ ]:
f1['weights']

In [ ]:
for i,j in enumerate(anacats):
    print(i, j)

In [ ]:
lumi = {
    "2016APV": 19800.,
    "2016": 16120., #35920 - 19800
    "2016all": 35920,
    "2017": 41530.,
    "2018": 59800., #59740./10., #Blinding
    "Full": 137190.
}

def MakeCMSLabel(AX, CatInt, IOV, isData=True):
    dytext = ''
    if 'cen' in anacats[CatInt]:
        dytext = r'$\Delta y$ < 1.0'
    elif 'fwd' in anacats[CatInt]:
        dytext = r'$\Delta y$ > 1.0'

    btext = ''
    if '0b' in anacats[CatInt]:
        btext = '0 b-tags'
    elif '1b' in anacats[CatInt]:
        btext = '1 b-tag'
    elif '2b' in anacats[CatInt]:
        btext = '2 b-tags'
        
    text = f'Private:    {btext}, {dytext}'
    
    hep.cms.label(text, data=False, lumi='{0:0.1f}'.format(lumi[IOV]/1000.), loc=0, fontsize=17, ax=AX)

In [ ]:
# Syst = ['prefiring', 'pileup', 'pdf', 'q2', 'btag', 'jes', 'jer', 'toptagxs', 'toptagsf', 'lumi']

Syst = 'toptagsf'
iov = '2016all'

avgUp = 0.
avgDown = 0.
avgNom = 0.
for icat in range(12, 18):
    fig, (ax, rax) = plt.subplots(nrows=2, height_ratios=[3, 1])
    
    if 'all' in iov:
        Nom_apv = getHist('ttbarmass', 'TTbar', False, '2016APV', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':'nominal'})
        Up_apv = getHist('ttbarmass', 'TTbar', False, '2016APV', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':f'{Syst}Up'})
        Down_apv = getHist('ttbarmass', 'TTbar', False, '2016APV', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':f'{Syst}Down'})
        Nom_noapv = getHist('ttbarmass', 'TTbar', False, '2016', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':'nominal'})
        Up_noapv = getHist('ttbarmass', 'TTbar', False, '2016', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':f'{Syst}Up'})
        Down_noapv = getHist('ttbarmass', 'TTbar', False, '2016', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':f'{Syst}Down'})
        
        Nom = Nom_apv + Nom_noapv
        Up = Up_apv + Up_noapv
        Down = Down_apv + Down_noapv
    
    S = hist.Stack(Up, Nom, Down)
    S.plot(ax=ax, stack=False, histtype="step", lw=3, label=['Up','Nom.','Down'], color=['blue', 'green', 'red'])
    
    ax.set_ylim(bottom=0.)
    ax.set_xlabel('')
    ax.set_xlim(800, 6000)
    ax.legend()
    
    MakeCMSLabel(ax, icat, iov, False)
    
    UpCorr =  Up / Nom.values()
    DownCorr =  Down / Nom.values()
    
    UpCorrArr = np.where(np.isnan(np.array(UpCorr.values())), 1., UpCorr.values())
    DownCorrArr = np.where(np.isnan(np.array(DownCorr.values())), 1., DownCorr.values())
    
    print('Nominal weight = ', "{:.2f}".format(np.abs(np.sum(Nom.values())/len(Nom.values()))))
    print( '+', "{:.1f}".format(np.abs(1. - (np.abs(np.sum(UpCorrArr)/len(UpCorrArr))))*100.), '%' )
    print( '-', "{:.1f}".format(np.abs(1. - (np.abs(np.sum(DownCorrArr)/len(DownCorrArr))))*100.), '%' )
    
    avgNom += np.abs(np.sum(Nom.values())/len(Nom.values()))
    avgUp += np.abs((1. - (np.abs(np.sum(UpCorrArr)/len(UpCorrArr))))*100.)
    avgDown += np.abs((1. - (np.abs(np.sum(DownCorrArr)/len(DownCorrArr))))*100.)
    
    ratioUp = hep.histplot(UpCorr, ax=rax, histtype='step', color='blue')
    ratioDown = hep.histplot(DownCorr, ax=rax, histtype='step', color='red')
    
    rax.set_ylim(0.75,1.25)
    rax.axhline(1, color='green')
    rax.set_ylabel('Syst./Nom.')
    rax.set_xlim(800, 6000)
    
    # if Syst == 'toptagsf': rax.set_ylim(0.5,2)
    # elif Syst == 'lumi': rax.set_ylim(0.95,1.05)
    # elif Syst == 'prefiring': rax.set_ylim(0.95,1.05)
    # elif Syst == 'pileup': rax.set_ylim(0.80,1.20)
    # elif Syst == 'pdf': rax.set_ylim(0.80,1.20)
    # elif Syst == 'q2': rax.set_ylim(0.20,1.80)
    # elif Syst == 'btag': rax.set_ylim(0.75,1.25)
    # elif Syst == 'toptagxs': rax.set_ylim(0.80,1.20)
    # elif Syst == 'jer': rax.set_ylim(0.80,1.20)
        
    
    leg = plt.text(0.73, 0.55, f'TTbar Sim.\nSignal Region\n{Syst} correction',
                    fontsize=16,
                    weight='bold',
                    transform=ax.transAxes
                   )
    plt.legend()
    plt.show()
    
print('\n\nAverages:\n----------------')
print('Nominal Weight = ', "{:.2f}".format(avgNom/6.))
print('+', "{:.1f}".format(avgUp/6.), '%')
print('-', "{:.1f}".format(avgDown/6.), '%')

In [ ]:
# for icat in range(12, 17):
#     fig = plt.figure(icat+1)
    
#     f1['ttbarmass_fine'][{'anacat':icat, 'systematic':'q2Up'}].project('ttbarmass').plot(label='Up')
#     f1['ttbarmass_fine'][{'anacat':icat, 'systematic':'nominal'}].project('ttbarmass').plot(label='Nom.')
#     f1['ttbarmass_fine'][{'anacat':icat, 'systematic':'q2Down'}].project('ttbarmass').plot(label='Down')
    
#     plt.title("APV " + anacats[icat])
#     plt.legend()

In [ ]:
# for icat in range(17):
#     fig = plt.figure(icat+1)
#     f1['ttbarmass_bare'].project("anacat", "ttbarmass")[icat,:].plot(density=True, label="Bare")
#     f1['ttbarmass_fine'].project("anacat", "ttbarmass")[icat,:].plot(density=True, label="Weighted")
#     plt.title("APV " + anacats[icat])
#     plt.legend()

# Full Luminosity

In [ ]:
# Syst = ['prefiring', 'pileup', 'pdf', 'q2', 'btag', 'jes', 'jer', 'toptagxs', 'toptagsf', 'lumi']

Syst = 'lumi'
iov = 'Full'

avgUp = 0.
avgDown = 0.
avgNom = 0.
for icat in range(12, 18):
    fig, (ax, rax) = plt.subplots(nrows=2, height_ratios=[3, 1])
    
    if 'Full' in iov:
        Nom_apv = getHist('ttbarmass', 'TTbar', False, '2016APV', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':'nominal'})
        Up_apv = getHist('ttbarmass', 'TTbar', False, '2016APV', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':f'{Syst}Up'})
        Down_apv = getHist('ttbarmass', 'TTbar', False, '2016APV', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':f'{Syst}Down'})
        Nom_noapv = getHist('ttbarmass', 'TTbar', False, '2016', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':'nominal'})
        Up_noapv = getHist('ttbarmass', 'TTbar', False, '2016', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':f'{Syst}Up'})
        Down_noapv = getHist('ttbarmass', 'TTbar', False, '2016', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':f'{Syst}Down'})
        
        Nom_17 = getHist('ttbarmass', 'TTbar', False, '2017', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':'nominal'})
        Up_17 = getHist('ttbarmass', 'TTbar', False, '2017', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':f'{Syst}Up'})
        Down_17 = getHist('ttbarmass', 'TTbar', False, '2017', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':f'{Syst}Down'})
        Nom_18 = getHist('ttbarmass', 'TTbar', False, '2018', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':'nominal'})
        Up_18 = getHist('ttbarmass', 'TTbar', False, '2018', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':f'{Syst}Up'})
        Down_18 = getHist('ttbarmass', 'TTbar', False, '2018', sum_axes=[], integrate_axes={'anacat':icat, 'systematic':f'{Syst}Down'})
        
        Nom = Nom_apv + Nom_noapv + Nom_17 + Nom_18
        Up = Up_apv + Up_noapv + Up_17 + Up_18
        Down = Down_apv + Down_noapv + Down_17 + Down_18
        
    
    S = hist.Stack(Up, Nom, Down)
    S.plot(ax=ax, stack=False, histtype="step", lw=3, label=['Up','Nom.','Down'], color=['blue', 'green', 'red'])
    
    ax.set_ylim(bottom=0.)
    ax.set_xlabel('')
    ax.set_xlim(800, 6000)
    ax.legend()
    
    MakeCMSLabel(ax, icat, iov, False)
    
    UpCorr =  Up / Nom.values()
    DownCorr =  Down / Nom.values()
    
    UpIsBig = np.isnan(np.array(UpCorr.values())) | np.isinf(np.array(UpCorr.values())) # If up is either nan or inf, drop it when calculating the average weights and percent errors
    DownIsBig = np.isnan(np.array(DownCorr.values())) | np.isinf(np.array(DownCorr.values())) # If down is either nan or inf, drop it when calculating the average weights and percent errors
    
    UpCorrArr = np.where(UpIsBig, 1., UpCorr.values())
    DownCorrArr = np.where(DownIsBig, 1., DownCorr.values())
    
    ## ----- Only use the first several bins to avoid wild fluctuations in low statstics regions for higher masses ----- ##
    print('Nominal weight = ', "{:.2f}".format(np.abs(np.sum(Nom[:8].values())/len(Nom[:8].values()))))
    print( '+', "{:.1f}".format(np.abs(1. - (np.abs(np.sum(UpCorrArr[:8])/len(UpCorrArr[:8]))))*100.), '%' )
    print( '-', "{:.1f}".format(np.abs(1. - (np.abs(np.sum(DownCorrArr[:8])/len(DownCorrArr[:8]))))*100.), '%' )
    # print(UpCorrArr)
    # print(DownCorrArr)
    
    ## ----- Only use the first several bins to avoid wild fluctuations in low statstics regions for higher masses ----- ##
    avgNom += np.abs(np.sum(Nom[:8].values())/len(Nom[:8].values()))
    avgUp += np.abs((1. - (np.abs(np.sum(UpCorrArr[:8])/len(UpCorrArr[:8]))))*100.)
    avgDown += np.abs((1. - (np.abs(np.sum(DownCorrArr[:8])/len(DownCorrArr[:8]))))*100.)
    
    ratioUp = hep.histplot(UpCorr, ax=rax, histtype='step', color='blue')
    ratioDown = hep.histplot(DownCorr, ax=rax, histtype='step', color='red')
    
    rax.set_ylim(0.65,1.35) #rax.set_ylim(0.35,1.65) 
    rax.axhline(1, color='green')
    rax.set_ylabel('Syst./Nom.')
    rax.set_xlim(800, 6000)
    
    # if Syst == 'toptagsf': rax.set_ylim(0.5,2)
    # elif Syst == 'lumi': rax.set_ylim(0.95,1.05)
    # elif Syst == 'prefiring': rax.set_ylim(0.95,1.05)
    # elif Syst == 'pileup': rax.set_ylim(0.80,1.20)
    # elif Syst == 'pdf': rax.set_ylim(0.80,1.20)
    # elif Syst == 'q2': rax.set_ylim(0.20,1.80)
    # elif Syst == 'btag': rax.set_ylim(0.75,1.25)
    # elif Syst == 'toptagxs': rax.set_ylim(0.80,1.20)
    # elif Syst == 'jer': rax.set_ylim(0.80,1.20)
        
    
    leg = plt.text(0.73, 0.55, f'TTbar Sim.\nSignal Region\n{Syst} correction',
                    fontsize=16,
                    weight='bold',
                    transform=ax.transAxes
                   )
    plt.legend()
    plt.show()
    
print('\n\nAverages:\n----------------')
print('Nominal Weight = ', "{:.2f}".format(avgNom/6.))
print('+', "{:.1f}".format(avgUp/6.), '%')
print('-', "{:.1f}".format(avgDown/6.), '%')